# Station Training Baseline — KDAL: Dallas Love Field

**Status: active baseline.** This is the complete per-station workflow: the **KDAL V20 no-peak aligned** point pipeline, followed in this same notebook by **Ordinal Probabilities Model 2**, the pure cumulative-threshold ordinal probability model. Versioned source notebooks remain reference-only; new station work starts here.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_baseline_kdal_no_peak_stack"
EXPORT_MODEL_WEIGHTS = True
EXPORT_LIVE_MODEL_WEIGHTS = False
POINT_EVALUATION_TRAIN_YEARS = (2021, 2025)
POINT_BUCKET_CONTRACT = "polymarket_half_up_2f"
POINT_MAX_FEATURE_MISSING_FRACTION = 0.03
LIVE_POINT_MODEL_VERSION = "station_high_regressor_live_kdal_no_peak_stack_2026"
PROBABILITY_MODEL_VERSION = "station_bucket_baseline_kdal_no_peak_ordinal_pure"
PROBABILITY_FEATURE_PROFILE = "common_no_peak"
PROBABILITY_FEATURE_COUNT = 59
PROBABILITY_PROVIDERS = ('gfs', 'hrrr', 'nbm')
PROBABILITY_DEVELOPMENT_YEARS = (2023, 2024, 2025)
PROBABILITY_FORWARD_VALIDATION_YEARS = (2024, 2025)
PROBABILITY_HOLDOUT_YEAR = 2026
PROJECT_ROOT


In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## Point-model contract

`feature_version="v11_settlement_fix_temp"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,2021,2021-01-01,2026-07-29
7,KDAL,hrrr,2027,2021-01-01,2026-07-29
8,KDAL,nbm,2015,2021-01-01,2026-07-29


## Model Scores


In [ ]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11_settlement_fix_temp",
    training_profile="v20_aligned",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=POINT_MAX_FEATURE_MISSING_FRACTION,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_training_baseline" / "KDAL",
)

config.resolved_optuna_storage_path()


In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-07-31 19:34:03,013] Using an existing study with name 'KDAL_v11_settlement_fix_temp_remaining_warmup_v20_aligned_base_xgboost_mae_f_wide' instead of creating a new one.
[I 2026-07-31 19:34:03,245] Using an existing study with name 'KDAL_v11_settlement_fix_temp_remaining_warmup_v20_aligned_base_lightgbm_mae_f_wide' instead of creating a new one.
[I 2026-07-31 19:34:03,485] Using an existing study with name 'KDAL_v11_settlement_fix_temp_remaining_warmup_v20_aligned_base_catboost_mae_f_wide' instead of creating a new one.
[I 2026-07-31 19:57:05,076] Using an existing study with name 'KDAL_v11_settlement_fix_temp_remaining_warmup_v20_aligned_stack_ridge_stack_mae_f_wide' instead of creating a new one.


,period,method,count,mae_f,rmse_f
0,validation_2022_2025,xgboost,1204,1.385073,1.894045
1,validation_2022_2025,lightgbm,1204,1.431259,1.942772
2,validation_2022_2025,catboost,1204,1.396631,1.935174
3,validation_2022_2025,provider_mean,1204,2.558763,3.456457
4,validation_2022_2025,provider_median,1204,2.266280,3.219940
5,validation_2022_2025,nbm_raw,1204,2.060784,2.996452
6,validation_2022_2025,hrrr_raw,1204,4.527251,5.558282
7,validation_2022_2025,gfs_raw,1204,2.939431,3.959690
8,test_2026,xgboost,188,1.269613,1.712286
9,test_2026,lightgbm,188,1.288899,1.713130


## Point-model market-bucket hit rate

This score belongs to the continuous point model, not the ordinal probability
model. The configured market contract is `polymarket_half_up_2f`: two-degree Fahrenheit bracket after half-up degree rounding.
The forward score is honest chronological evidence. The holdout score is shown
separately and remains exploratory.


In [ ]:
from src.calibration.temperature_buckets import (
    point_bucket_metrics,
    point_bucket_predictions,
)
from src.calibration.v19_bucket import crossfit_ridge_predictions

point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
point_forward_predictions = point_forward_predictions.loc[
    point_forward_predictions["validation_year"].isin(
        PROBABILITY_FORWARD_VALIDATION_YEARS
    )
].copy()
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

point_forward_bucket_predictions = point_bucket_predictions(
    point_forward_predictions,
    POINT_BUCKET_CONTRACT,
)
point_forward_bucket_metrics = point_bucket_metrics(
    point_forward_predictions,
    POINT_BUCKET_CONTRACT,
)
point_forward_bucket_metrics["evaluation_status"] = "honest_forward"
point_forward_bucket_metrics


In [ ]:
point_holdout_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not point_holdout_predictions.empty

point_holdout_bucket_predictions = point_bucket_predictions(
    point_holdout_predictions,
    POINT_BUCKET_CONTRACT,
)
point_holdout_bucket_metrics = point_bucket_metrics(
    point_holdout_predictions,
    POINT_BUCKET_CONTRACT,
)
point_holdout_bucket_metrics["evaluation_status"] = "exploratory_holdout"
point_holdout_bucket_metrics


In [ ]:
point_bucket_output_dir = config.resolved_output_dir() / "point_bucket_evaluation"
point_bucket_output_dir.mkdir(parents=True, exist_ok=True)
point_forward_bucket_predictions.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_forward_predictions.csv", index=False
)
point_forward_bucket_metrics.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_forward_metrics.csv", index=False
)
point_holdout_bucket_predictions.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_predictions.csv",
    index=False,
)
point_holdout_bucket_metrics.to_csv(
    point_bucket_output_dir / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_holdout_metrics.csv",
    index=False,
)
point_bucket_output_dir


In [ ]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        train_years=POINT_EVALUATION_TRAIN_YEARS,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        bucket_contract=POINT_BUCKET_CONTRACT,
        source_pipeline="notebooks/station_training_baseline/stations/KDAL",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")

if EXPORT_MODEL_WEIGHTS:
    import json as _point_export_json

    evaluation_point_manifest = _point_export_json.loads(
        exported_weights.manifest_path.read_text(encoding="utf-8")
    )
    assert evaluation_point_manifest["model_version"] == MODEL_VERSION
    assert evaluation_point_manifest["training"]["train_start_year"] == POINT_EVALUATION_TRAIN_YEARS[0]
    assert evaluation_point_manifest["training"]["train_end_year"] == POINT_EVALUATION_TRAIN_YEARS[1]
    assert evaluation_point_manifest["model_contract"]["max_feature_missing_fraction"] == POINT_MAX_FEATURE_MISSING_FRACTION
    assert evaluation_point_manifest["model_contract"]["bucket_contract"] == POINT_BUCKET_CONTRACT
    assert all(
        row["missing_fraction"] <= POINT_MAX_FEATURE_MISSING_FRACTION
        for row in evaluation_point_manifest["features"]["missingness_audit"]
        if row["selected"]
    )


## Optional live-production point bundle

The evaluation bundle above is frozen before the exploratory holdout and is the
only point bundle used by probability training and holdout reporting. A live
production refit may use all completed actuals, including completed holdout-year
dates, but it must reuse the evaluation bundle's exact ordered feature contract.
The exporter fails closed if any frozen feature is absent or violates the refit
missingness guard; it never adds newly eligible columns. The live refit has a
distinct version and cannot claim holdout performance as out-of-sample evidence.
Keep this export disabled until the source is committed; then create the
immutable release record in a separate promotion review.


In [ ]:
live_exported_weights = None
if EXPORT_LIVE_MODEL_WEIGHTS:
    assert EXPORT_MODEL_WEIGHTS
    frozen_point_feature_names = tuple(evaluation_point_manifest["features"]["all"])
    frozen_point_feature_source = {
        "kind": "evaluation_manifest",
        "model_version": evaluation_point_manifest["model_version"],
        "bundle_sha256": evaluation_point_manifest["artifact_integrity"]["bundle_sha256"],
        "ordered_features_sha256": evaluation_point_manifest["features"]["ordered_features_sha256"],
    }
    live_exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID if "CITY_ID" in globals() else None,
        artifact_dir=config.resolved_output_dir(),
        model_version=LIVE_POINT_MODEL_VERSION,
        train_years=None,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        max_feature_missing_fraction=config.effective_max_feature_missing_fraction,
        bucket_contract=POINT_BUCKET_CONTRACT,
        source_pipeline="notebooks/station_training_baseline/stations/KDAL",
        frozen_feature_names=frozen_point_feature_names,
        feature_contract_source=frozen_point_feature_source,
    )
    assert live_exported_weights.bundle_path != exported_weights.bundle_path
    assert LIVE_POINT_MODEL_VERSION != MODEL_VERSION
    live_point_manifest = _point_export_json.loads(
        live_exported_weights.manifest_path.read_text(encoding="utf-8")
    )
    assert live_point_manifest["features"]["selection_mode"] == "frozen_evaluation_contract"
    assert live_point_manifest["features"]["all"] == list(frozen_point_feature_names)
    assert live_point_manifest["features"]["feature_count"] == len(frozen_point_feature_names)
    assert (
        live_point_manifest["features"]["ordered_features_sha256"]
        == evaluation_point_manifest["features"]["ordered_features_sha256"]
    )
    print(
        "Live bundle exported as an unreleased candidate. "
        "Create a clean-checkout release record before promotion."
    )
else:
    print("Live-production export disabled; evaluation bundle remains frozen.")


## Point-model feature coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v4_observed_precip_any,100.000000
1,v4_forecast_wet_observed_dry,100.000000
2,v4_forecast_observed_precip_match,100.000000
3,v4_all_forecast_precip,100.000000
4,v4_any_forecast_precip,100.000000
5,climatology_high_10y_std_f,100.000000
6,climatology_high_10y_f,100.000000
7,climatology_high_10y_count,100.000000
8,v8_month_remaining_warmup_count,100.000000
9,v4_observed_wet_forecast_dry,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
165,v2_recent_heat_anomaly_f,numeric
166,v2_recent_heat_momentum_f,numeric
167,v2_morning_warmup_to_consensus_f,numeric
168,v2_consensus_minus_7d_actual_f,numeric
169,v2_spread_per_warmup_f,numeric
170,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,99.557957
1,observed_temp_change_last_3h_f,99.557957
2,observed_morning_warmup_rate_f_per_hour,99.557957
3,observed_high_so_far_change_since_9am_f,99.557957


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="holdout_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
8,holdout_2026,xgboost,188,130,69.148936
3,holdout_2026,lightgbm,188,127,67.553191
7,holdout_2026,ridge_stack,188,123,65.425532
0,holdout_2026,catboost,188,119,63.297872
4,holdout_2026,nbm_raw,188,89,47.340426
6,holdout_2026,provider_median,188,77,40.957447
1,holdout_2026,gfs_raw,188,74,39.361702
5,holdout_2026,provider_mean,188,65,34.574468
2,holdout_2026,hrrr_raw,188,26,13.829787
16,validation_2024_2025,xgboost,1204,789,65.531561


## 2026 Exploratory Holdout Weather Brackets


In [14]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,188,1.269613,1.712286,1.340807,46.276596,3.290053,1.063830
1,lightgbm,188,1.288899,1.713130,1.336199,47.340426,3.314961,0.531915
2,catboost,188,1.364060,1.776818,1.344758,43.085106,3.530457,1.063830
3,ridge_stack,188,1.293793,1.717956,1.316902,47.87234,3.472096,0.531915
4,provider_mean,188,2.518372,3.215513,1.684033,23.404255,5.303652,7.446809
5,provider_median,188,2.373275,3.134393,1.745988,23.93617,5.756985,7.978723
6,nbm_raw,188,2.087807,2.810436,1.728210,32.978723,5.756985,7.978723
7,hrrr_raw,188,4.668649,5.389658,1.838992,7.446809,9.135173,42.021277
8,gfs_raw,188,2.591690,3.470045,1.966137,26.595745,6.397990,12.765957


## Train-Fold 3% Missingness Audit


In [15]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2022_to_2023     False             43                  1.000000
 1  fold_2021_2022_to_2023      True            187                  0.016736
 2  fold_2021_2023_to_2024     False             43                  1.000000
 3  fold_2021_2023_to_2024      True            187                  0.019815
 4  fold_2021_2024_to_2025     False             32                  1.000000
 5  fold_2021_2024_to_2025      True            198                  0.023723
 6       fold_2021_to_2022     False             43                  1.000000
 7       fold_2021_to_2022      True            187                  0.024691
 8    test_refit_2021_2025     False             32                  1.000000
 9    test_refit_2021_2025      True            198                  0.019350,
                         fold  train_start_year  train_end_year  \
 254   fold_2021_2022_to_2023              2021            2022   
 255   

## Expanded 11 AM Feature Coverage and Provider Count


In [16]:
new_feature_coverage = (
    result.features[V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v11sf_forecast_temp_11am_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_11am_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                     v11sf_forecast_temp_11am_mean_f     99.557957
 1                   v11sf_forecast_temp_11am_median_f     99.557957
 2           v11sf_forecast_temp_11am_minus_observed_f     99.557957
 3                v11sf_forecast_temp_11am_abs_error_f     99.557957
 4               v11sf_forecast_temp_11am_warm_error_f     99.557957
 5               v11sf_forecast_temp_11am_cool_error_f     99.557957
 6                   v11sf_forecast_temp_11am_spread_f     99.557957
 7             v11sf_forecast_temp_11am_provider_count    100.000000
 8   v11sf_forecast_temp_bias_remaining_warmup_inte...     99.557957
 9          v11sf_observation_adjusted_provider_high_f     99.557957
 10                 v11sf_forecast_warmup_after_11am_f     99.557957,
    available_provider_count  row_count    row_pct
 0                         0          9   0.442043
 1                         1         23   1.129666
 2                

## New-Feature Importance


In [17]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V11_SETTLEMENT_FIX_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
18,catboost,trial_25,v11sf_forecast_warmup_after_11am_f,0.038685,0.011297,10,2021,2025,2026,1447,188
83,catboost,trial_25,v11sf_forecast_temp_11am_abs_error_f,0.006668,0.005839,10,2021,2025,2026,1447,188
84,catboost,trial_25,v11sf_forecast_temp_11am_minus_observed_f,0.006306,0.002083,10,2021,2025,2026,1447,188
107,catboost,trial_25,v11sf_forecast_temp_11am_warm_error_f,0.004467,0.001968,10,2021,2025,2026,1447,188
126,catboost,trial_25,v11sf_forecast_temp_11am_mean_f,0.003342,0.002410,10,2021,2025,2026,1447,188
129,catboost,trial_25,v11sf_forecast_temp_11am_cool_error_f,0.003291,0.002294,10,2021,2025,2026,1447,188
156,catboost,trial_25,v11sf_observation_adjusted_provider_high_f,0.002468,0.016815,10,2021,2025,2026,1447,188
158,catboost,trial_25,v11sf_forecast_temp_11am_median_f,0.002450,0.003441,10,2021,2025,2026,1447,188
258,catboost,trial_25,v11sf_forecast_temp_11am_spread_f,0.000689,0.005432,10,2021,2025,2026,1447,188
368,catboost,trial_25,v11sf_forecast_temp_11am_provider_count,0.000000,0.000000,10,2021,2025,2026,1447,188


## 2026 Exploratory Holdout Monthly Metrics


In [18]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,1.352152,1.985362,0.540681
1,catboost,2,28,1.308246,1.638277,0.231716
2,catboost,3,30,1.471600,1.834256,0.898627
3,catboost,4,29,1.053317,1.340813,-0.162962
4,catboost,5,31,1.505765,1.948824,-0.345897
...,...,...,...,...,...,...
58,xgboost,3,30,1.087257,1.467405,0.272979
59,xgboost,4,29,1.175928,1.564155,-0.334330
60,xgboost,5,31,1.422158,1.937706,-0.163833
61,xgboost,6,21,1.414767,1.862956,-0.167408


## Performance by Warm/Cool 11 AM Forecast Delta


In [19]:
delta_by_date = result.features[[
    "contract_date",
    "v11sf_forecast_temp_11am_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v11sf_forecast_temp_11am_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,51,1.262075,0.179362
1,catboost,cool_0.5_to_2f,64,1.333314,0.105274
2,catboost,near_match,32,1.233407,0.353728
3,catboost,warm_0.5_to_2f,32,1.492293,0.488546
4,catboost,warm_gt_2f,9,2.169214,-2.169214
5,gfs_raw,cool_gt_2f,51,2.160618,1.774472
6,gfs_raw,cool_0.5_to_2f,64,2.578223,0.423649
7,gfs_raw,near_match,32,1.833364,-0.003580
8,gfs_raw,warm_0.5_to_2f,32,2.830187,-0.521679
9,gfs_raw,warm_gt_2f,9,6.978478,-2.009305


## Ordinal Probabilities Model 2 — part of this station run

This is the station's verified **pure ordinal** setup, not a separate experiment:

- target: rounded actual degree minus rounded point-model degree;
- ordered classes: `≤-4, -3, -2, -1, 0, +1, +2, +3, ≥+4`;
- eight cumulative logistic regressions with median imputation and scaling;
- forced family `ordinal_logistic`;
- learned-model blend weight `1.0` (no empirical probability blend);
- development years: `[2023, 2024, 2025]`;
- honest forward-validation years: `[2024, 2025]`, each trained only on
  earlier development years;
- the last 90 days of each outer training period tune regularization,
  class weighting, and temperature;
- final probability artifact fitted on all available development rows;
- 2026 is an exploratory holdout,
  **not** an out-of-fold training fold.

Preprocessing is fitted independently inside each chronological training fold.
Continuous features are standardized after median imputation. The verified
implementation does not apply skew transforms or feature
winsorization/clipping.

Degree probabilities are aggregated into the actual 2°F market buckets after
prediction, so bucket boundaries do not need to be centered on the point degree.
The ordinal output remains shadow/research-only until fresh promotion gates pass.


In [20]:
import json

from src.calibration.bucket_probability import (
    build_probability_frame,
    default_candidate_specs,
    evaluate_probability_holdout,
    export_probability_bundle,
    fit_probability_system,
    probability_feature_names,
    probability_metrics,
    sha256_file,
)
from src.calibration.v19_bucket import crossfit_ridge_predictions


### Exact ordinal-regression feature contract


In [21]:
ordinal_feature_contract = pd.DataFrame(
    {
        "position": range(
            1,
            len(
                probability_feature_names(
                    include_peak_features=False,
                    feature_profile=PROBABILITY_FEATURE_PROFILE,
                )
            )
            + 1,
        ),
        "feature": probability_feature_names(
            include_peak_features=False,
            feature_profile=PROBABILITY_FEATURE_PROFILE,
        ),
    }
)
ordinal_feature_contract


,position,feature
0,1,point_prediction_f
1,2,rounded_point_degree_f
2,3,point_rounding_remainder_f
3,4,point_distance_to_round_boundary_f
4,5,point_signed_distance_to_round_boundary_f
5,6,xgboost_predicted_high_f
6,7,lightgbm_predicted_high_f
7,8,catboost_predicted_high_f
8,9,base_prediction_mean_f
9,10,base_prediction_spread_f


### Fit with chronological [2024, 2025] outer validation


In [22]:
point_forward_predictions = crossfit_ridge_predictions(
    result.validation_predictions,
    providers=PROBABILITY_PROVIDERS,
)
assert not point_forward_predictions.empty
assert (
    point_forward_predictions["train_through_year"]
    < point_forward_predictions["validation_year"]
).all()

ordinal_training_frame = build_probability_frame(
    result.features,
    point_forward_predictions,
    result.validation_predictions,
    include_peak_features=False,
    feature_profile=PROBABILITY_FEATURE_PROFILE,
)
ordinal_candidate_specs = [
    spec
    for spec in default_candidate_specs()
    if spec.family in {"empirical", "ordinal_logistic"}
]
ordinal_bundle, ordinal_forward_predictions, ordinal_tuning = (
    fit_probability_system(
        ordinal_training_frame,
        station_id=STATION_ID,
        point_model_version=MODEL_VERSION,
        point_bundle_sha256=sha256_file(exported_weights.bundle_path),
        include_peak_features=False,
        feature_profile=PROBABILITY_FEATURE_PROFILE,
        model_version=PROBABILITY_MODEL_VERSION,
        candidate_specs=ordinal_candidate_specs,
        forced_family="ordinal_logistic",
        blend_weights=(1.0,),
        development_years=PROBABILITY_DEVELOPMENT_YEARS,
        forward_validation_years=PROBABILITY_FORWARD_VALIDATION_YEARS,
    )
)
assert ordinal_bundle["selected_family"] == "ordinal_logistic"
assert ordinal_bundle["blend_weight"] == 1.0
probability_metrics(ordinal_forward_predictions)


,log_loss,brier,ranked_probability_score,offset_accuracy,top_two_accuracy,calibration_error,count
0,1.997115,0.829218,0.11452,0.234783,0.482609,0.030541,690


### Verify chronology and the frozen probability contract


In [23]:
ordinal_forward_dates = pd.to_datetime(
    ordinal_forward_predictions["contract_date"]
)
assert set(ordinal_forward_predictions["validation_year"]) == set(
    PROBABILITY_FORWARD_VALIDATION_YEARS
)
assert (
    pd.to_datetime(ordinal_forward_predictions["model_training_cutoff"])
    < ordinal_forward_dates
).all()
assert (
    pd.to_datetime(
        ordinal_forward_predictions["calibration_training_cutoff"]
    )
    < pd.to_datetime(
        ordinal_forward_predictions["calibration_validation_start"]
    )
).all()
assert (
    pd.to_datetime(
        ordinal_forward_predictions["calibration_validation_cutoff"]
    )
    < ordinal_forward_dates
).all()
assert ordinal_bundle["selected_family"] == "ordinal_logistic"
assert ordinal_bundle["family_selection_mode"] == "forced"
assert ordinal_bundle["blend_weight"] == 1.0
assert ordinal_bundle["feature_profile"] == PROBABILITY_FEATURE_PROFILE
assert len(ordinal_bundle["feature_names"]) == PROBABILITY_FEATURE_COUNT
assert not any("peak" in name.lower() for name in ordinal_bundle["feature_names"])
{
    "development_years": list(PROBABILITY_DEVELOPMENT_YEARS),
    "forward_validation_years": sorted(
        ordinal_forward_predictions["validation_year"].unique().tolist()
    ),
    "final_training_start": ordinal_bundle["training_start"],
    "final_training_cutoff": ordinal_bundle["training_cutoff"],
}


{'development_years': [2023, 2024, 2025],
 'forward_validation_years': [2024, 2025],
 'final_training_start': '2023-01-01',
 'final_training_cutoff': '2025-12-31'}

### Evaluate the frozen ordinal model on the 2026 holdout


In [24]:
holdout_point_predictions = result.test_predictions.loc[
    result.test_predictions["method"].eq("ridge_stack"),
    ["contract_date", "actual_high_f", "predicted_high_f"],
].copy()
assert not holdout_point_predictions.empty

ordinal_holdout_predictions, ordinal_holdout_metrics = (
    evaluate_probability_holdout(
        result.features,
        holdout_point_predictions,
        result.test_predictions,
        ordinal_bundle,
        holdout_year=PROBABILITY_HOLDOUT_YEAR,
    )
)
assert not ordinal_holdout_predictions.empty
assert not ordinal_holdout_metrics.empty
for probability_column in (
    "offset_probabilities",
    "degree_probabilities",
    "bucket_probabilities",
):
    assert ordinal_holdout_predictions[probability_column].map(
        lambda probabilities: np.isclose(
            sum(float(value) for value in probabilities.values()),
            1.0,
            atol=1e-10,
        )
    ).all()

ordinal_bundle["holdout_metrics"] = ordinal_holdout_metrics.iloc[0].to_dict()
ordinal_bundle["holdout_status"] = "exploratory"
ordinal_bundle["historical_acceptance"] = {
    "passed": False,
    "reasons": ["fresh_shadow_data_required"],
    "holdout_status": "exploratory_previously_inspected",
}
ordinal_holdout_metrics


,holdout_year,count,offset_log_loss,multiclass_brier,ranked_probability_score,offset_accuracy,offset_top_two_accuracy,bucket_log_loss,point_bucket_accuracy,probability_bucket_accuracy,switch_count
0,2026,188,2.02477,0.819907,0.113704,0.271277,0.5,1.27685,0.478723,0.452128,30


### Export point and probability outputs together


In [25]:
probability_output_dir = config.resolved_output_dir() / "ordinal_probability"
probability_output_dir.mkdir(parents=True, exist_ok=True)

serializable_forward = ordinal_forward_predictions.copy()
serializable_forward["offset_probabilities"] = (
    serializable_forward["offset_probabilities"].map(
        lambda value: json.dumps(value, sort_keys=True)
    )
)
serializable_forward.to_csv(
    probability_output_dir / f"{STATION_ID}_forward_probability_predictions.csv",
    index=False,
)
ordinal_tuning.to_csv(
    probability_output_dir / f"{STATION_ID}_probability_tuning.csv",
    index=False,
)
probability_metrics(ordinal_forward_predictions).to_csv(
    probability_output_dir / f"{STATION_ID}_forward_probability_metrics.csv",
    index=False,
)

serializable_holdout = ordinal_holdout_predictions.copy()
for column in (
    "offset_probabilities",
    "degree_probabilities",
    "bucket_probabilities",
):
    serializable_holdout[column] = serializable_holdout[column].map(
        lambda value: json.dumps(value, sort_keys=True)
    )
serializable_holdout.to_csv(
    probability_output_dir
    / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_probability_holdout_predictions.csv",
    index=False,
)
ordinal_holdout_metrics.to_csv(
    probability_output_dir
    / f"{STATION_ID}_{PROBABILITY_HOLDOUT_YEAR}_probability_holdout_metrics.csv",
    index=False,
)
ordinal_feature_contract.to_csv(
    probability_output_dir / f"{STATION_ID}_ordinal_feature_contract.csv",
    index=False,
)

ordinal_bundle_path, ordinal_manifest_path = export_probability_bundle(
    ordinal_bundle,
    probability_output_dir / "model_weights",
    source_identity={
        "pipeline": "station_training_baseline",
        "notebook": "notebooks/station_training_baseline/stations/KDAL/train_KDAL.ipynb",
        "point_workflow": "KDAL V20 no-peak aligned",
        "probability_setup": "Ordinal Probabilities Model 2",
    },
)
ordinal_manifest = json.loads(ordinal_manifest_path.read_text(encoding="utf-8"))
assert ordinal_manifest["point_bundle_sha256"] == sha256_file(
    exported_weights.bundle_path
)
assert ordinal_manifest["artifact_integrity"]["bundle_sha256"] == sha256_file(
    ordinal_bundle_path
)
{
    "point_bundle": exported_weights.bundle_path,
    "point_manifest": exported_weights.manifest_path,
    "ordinal_bundle": ordinal_bundle_path,
    "ordinal_manifest": ordinal_manifest_path,
    "output_dir": probability_output_dir,
}


{'point_bundle': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/KDAL/model_weights/KDAL_station_high_regressor_baseline_kdal_no_peak_stack.joblib'),
 'point_manifest': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/KDAL/model_weights/KDAL_station_high_regressor_baseline_kdal_no_peak_stack.json'),
 'ordinal_bundle': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/KDAL/ordinal_probability/model_weights/KDAL_station_bucket_baseline_kdal_no_peak_ordinal_pure.joblib'),
 'ordinal_manifest': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/KDAL/ordinal_probability/model_weights/KDAL_station_bucket_baseline_kdal_no_peak_ordinal_pure.json'),
 'output_dir': WindowsPath('D:/dev/weather-research/data/calibration/station_training_baseline/KDAL/ordinal_probability')}

## Required three-arm ordinal challenger export

This stage is part of the KDAL training run and freezes exactly three probability
arms in contract order:

1. best blended independent ordinal model (`model_weight < 1.0`);
2. best shared-slope ordinal model; and
3. best pure independent ordinal model (`model_weight = 1.0`).

Hyperparameters and feature ablations are selected only inside chronological
pre-2026 training folds. The already-inspected 2026 data remains exploratory.
Each arm exports a loadable `.joblib` weight bundle plus a JSON manifest whose
SHA-256 is checked below. These models remain shadow-only and never override the
V20 point-model bucket.


In [26]:
from scripts.run_kdal_ordinal_challenger_v1 import run_challenger
from src.calibration.kdal_ordinal_challenger import (
    FROZEN_CANDIDATE_ROLES,
)


### Train, evaluate, and export all three challenger weight bundles


In [27]:
challenger_run = run_challenger()
challenger_comparison = challenger_run["comparison"].copy()

assert tuple(challenger_comparison["candidate_role"]) == (
    FROZEN_CANDIDATE_ROLES
)
assert len(challenger_comparison) == 3
assert challenger_comparison.iloc[0]["model_weight"] < 1.0
assert (
    challenger_comparison.iloc[1]["family"]
    == "shared_slope_ordinal_logistic"
)
assert challenger_comparison.iloc[2]["model_weight"] == 1.0
assert len(challenger_run["bundle_paths"]) == 3
assert len(challenger_run["manifest_paths"]) == 3

for challenger_bundle_path, challenger_manifest_path in zip(
    challenger_run["bundle_paths"],
    challenger_run["manifest_paths"],
):
    assert challenger_bundle_path.is_file()
    assert challenger_manifest_path.is_file()
    challenger_manifest = json.loads(
        challenger_manifest_path.read_text(encoding="utf-8")
    )
    assert challenger_manifest["point_bundle_sha256"] == sha256_file(
        exported_weights.bundle_path
    )
    assert (
        challenger_manifest["artifact_integrity"]["bundle_sha256"]
        == sha256_file(challenger_bundle_path)
    )

challenger_comparison[
    [
        "candidate_role",
        "candidate_name",
        "family",
        "feature_set",
        "feature_count",
        "c",
        "temperature",
        "model_weight",
        "prior_strength",
        "bucket_log_loss",
        "holdout_bucket_log_loss",
        "manifest_path",
    ]
]


D:\dev\weather-research\src\calibration\kdal_ordinal_challenger.py:148: DtypeWarning: Columns (0: nbm_forecast_precip_intensity) have mixed types. Specify dtype option on import or set low_memory=False.
  features = pd.read_csv(features_path)


,candidate_role,candidate_name,family,feature_set,feature_count,c,temperature,model_weight,prior_strength,bucket_log_loss,holdout_bucket_log_loss,manifest_path
0,blended_ordinal,kdal_ordinal_challenger_v1_1_ordinal_logistic_...,ordinal_logistic,full_59,59,0.01,0.75,0.75,60.0,1.198634,1.291878,data\calibration\station_training_baseline\KDA...
1,shared_slope_ordinal,kdal_ordinal_challenger_v1_2_shared_slope_ordi...,shared_slope_ordinal_logistic,market_core_21,21,0.10,0.75,1.00,15.0,1.228515,1.327848,data\calibration\station_training_baseline\KDA...
2,pure_ordinal,kdal_ordinal_challenger_v1_3_ordinal_logistic_...,ordinal_logistic,full_59,59,0.01,1.00,1.00,15.0,1.200869,1.279097,data\calibration\station_training_baseline\KDA...


### Exported challenger artifacts


In [28]:
pd.DataFrame(
    {
        "candidate_role": FROZEN_CANDIDATE_ROLES,
        "weight_bundle": [
            str(path) for path in challenger_run["bundle_paths"]
        ],
        "manifest": [
            str(path) for path in challenger_run["manifest_paths"]
        ],
    }
)


,candidate_role,weight_bundle,manifest
0,blended_ordinal,D:\dev\weather-research\data\calibration\stati...,D:\dev\weather-research\data\calibration\stati...
1,shared_slope_ordinal,D:\dev\weather-research\data\calibration\stati...,D:\dev\weather-research\data\calibration\stati...
2,pure_ordinal,D:\dev\weather-research\data\calibration\stati...,D:\dev\weather-research\data\calibration\stati...
